# 5-10 공시 가져오기 — 종목별·키워드별 공시와 거래소 시장 공시

**질문 예시:** "삼성전자 최근 3개월 공시 목록 줘", "이번 주 유상증자 공시 낸 회사 뽑아줘",
"오늘 거래소가 낸 투자경고·공매도 과열 공시 보여줘"

DeepSearch 공시(`["company"],["disclosure"]`)에는 **DART 공시**(회사가 낸 공시)와 **KIND 공시**(거래소 시장본부·
시장감시위원회가 낸 시장조치)가 함께 들어 있습니다. 문서에는 제목·시각·회사·원문 링크만 있고 본문은 없습니다.
본문의 표·숫자가 필요하면 링크를 따라가 읽습니다 → [2-05 공시 본문](../2_disclosure_news/05_disclosure_text.ipynb).

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beaten-by-the-market/deepsearch/blob/main/scenarios/5_data_lookup/10_disclosure.ipynb)

## 0. 준비
Colab 왼쪽 🔑(Secrets)에 `DEEPSEARCH_API_KEY`를 등록하고 이 노트북의 액세스를 켠 뒤 아래 두 셀을 실행하세요. 두 번째 셀은 모든 시나리오에 공통인 헬퍼입니다(접어 두어도 됩니다).

In [1]:
# Colab에서만: 한글 차트 폰트 설치 (로컬에서는 건너뜀)
import sys
if "google.colab" in sys.modules:
    !pip -q install koreanize-matplotlib

In [2]:
# ── DeepSearch 헬퍼 (모든 시나리오 노트북 공통) ─────────────────────────────
# Colab: 왼쪽 🔑(Secrets)에 이름 DEEPSEARCH_API_KEY 로 키를 넣고 "노트북 액세스"를 켜세요.
#        키는 "아이디:비밀번호" 형식이든 base64 문자열이든 됩니다. 키 값은 출력하지 않습니다.
# 로컬: 환경변수 DEEPSEARCH_API_KEY / API_KEY 또는 상위 폴더의 .env 를 읽습니다.
import base64, json, os, sys, time
from datetime import date, datetime, timedelta

import pandas as pd
import requests

IN_COLAB = "google.colab" in sys.modules
API_URL = "https://api.deepsearch.com/v1/compute"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)


def _norm_key(k):
    k = k.strip().strip('"').strip("'")
    return base64.b64encode(k.encode()).decode() if ":" in k else k


def _find_key():
    if IN_COLAB:
        try:
            from google.colab import userdata
            k = userdata.get("DEEPSEARCH_API_KEY")
            if k:
                return _norm_key(k)
        except Exception:
            pass
    for name in ("DEEPSEARCH_API_KEY", "API_KEY"):
        if os.environ.get(name):
            return _norm_key(os.environ[name])
    d = os.path.abspath(os.getcwd())
    while True:
        p = os.path.join(d, ".env")
        if os.path.isfile(p):
            for line in open(p, encoding="utf-8-sig"):
                if "=" in line and not line.strip().startswith("#"):
                    k, v = line.split("=", 1)
                    if k.strip() in ("DEEPSEARCH_API_KEY", "API_KEY") and v.strip():
                        return _norm_key(v)
        if os.path.dirname(d) == d:
            break
        d = os.path.dirname(d)
    from getpass import getpass
    return _norm_key(getpass("DeepSearch API 키: "))


_KEY = _find_key()


class DSError(Exception):
    pass


def ds(query, timeout=180, retries=3):
    """compute 호출 → 응답 JSON. 서버가 돌려준 오류(data.exceptions)는 DSError."""
    last = None
    for i in range(retries):
        try:
            r = requests.post(API_URL, json={"input": query}, timeout=timeout,
                              headers={"Authorization": "Basic " + _KEY})
            body = r.json()
        except Exception as e:  # 네트워크·일시 오류만 재시도
            last = e
            time.sleep(3 * (i + 1))
            continue
        exc = (body.get("data") or {}).get("exceptions") or []
        if exc or r.status_code >= 400:
            msg = "; ".join(f"{x.get('class', '')}: {' '.join(map(str, x.get('args') or [x.get('message', '')]))}"
                            if isinstance(x, dict) else str(x) for x in exc)
            raise DSError(f"{r.status_code} {msg or body}"[:500])
        return body
    raise DSError(f"요청 실패: {last}")


def _pod(body):
    pods = (body.get("data") or {}).get("pods") or []
    return pods[1] if len(pods) >= 2 else {}


def ds_df(query, **kw):
    """DataFrame 결과(재무·시세·경제지표 등) → pandas DataFrame. 결과가 없으면 빈 DataFrame."""
    c = _pod(ds(query, **kw)).get("content") or {}
    data = c.get("data") if isinstance(c, dict) else None
    if not isinstance(data, dict) or not data:
        return pd.DataFrame()
    cols = [x for x in (c.get("columns") or data.keys()) if x in data]
    df = pd.DataFrame({x: data[x] for x in cols})
    for x in ("date", "created_at"):
        if x in df and df[x].dtype == object:
            df[x] = pd.to_datetime(df[x].astype(str).str[:19], errors="coerce")
    return df


def _polarity(x):
    """문서 긍부정: polarity.name, 없으면 customized_ml.kis 모델 값(최근 뉴스는 절반가량이 이쪽에만 있음)."""
    p = x.get("polarity")
    if isinstance(p, dict) and p.get("name"):
        return p["name"]
    k = (((x.get("customized_ml") or {}).get("kis") or {}).get("ip_multitask") or {}).get("polarity") or {}
    return k.get("name")


def doc_search(category, section, query, date_from, date_to, count=100, pages=1):
    """DocumentSearch → (총건수, 문서 DataFrame). category: news/research/company/patent. 기간은 1년 이내.
    종목 지정은 securities.symbol:005930 (접두어 없는 6자리)이 뉴스·리서치·공시 모두에서 가장 정확합니다.
    securities.symbol:KRX:005930 은 키워드 검색으로 바뀌어 건수가 줄고, securities.name: 은 리서치(공식명
    '에스케이하이닉스')·공시(DART명 '현대자동차')에서 놓칩니다(2026-11-25 확인). 여러 종목: securities.symbol:(005930 or 000660)"""
    rows, total = [], 0
    for page in range(1, pages + 1):
        q = (f'DocumentSearch(["{category}"],{json.dumps(section, ensure_ascii=False)},'
             f'{json.dumps(query, ensure_ascii=False)},count={count},page={page},'
             f'date_from={date_from},date_to={date_to})')
        c = _pod(ds(q)).get("content") or {}
        d = c.get("data") if isinstance(c.get("data"), dict) else c
        total = d.get("total_matches") or 0
        docs = d.get("docs") or []
        for x in docs:
            secs = x.get("securities") or []
            rows.append({
                "created_at": pd.to_datetime(str(x.get("created_at"))[:19], errors="coerce"),
                "publisher": x.get("publisher"),
                "title": " ".join(str(x.get("title") or "").split()),
                "stocks": ", ".join(s.get("name", "") for s in secs),
                "symbols": ", ".join(s.get("symbol", "") for s in secs),
                "polarity": _polarity(x),
                "url": x.get("content_url"),
            })
        if len(docs) < count:
            break
    cols = ["created_at", "publisher", "title", "stocks", "symbols", "polarity", "url"]
    return total, pd.DataFrame(rows, columns=cols)


def doc_trends(category, section, query, date_from, date_to, interval="1d"):
    """DocumentTrends → DataFrame(date, count)."""
    q = (f'DocumentTrends(["{category}"],{json.dumps(section, ensure_ascii=False)},'
         f'{json.dumps(query, ensure_ascii=False)},interval="{interval}",'
         f'date_from={date_from},date_to={date_to})')
    c = _pod(ds(q)).get("content") or {}
    d = c.get("data") if isinstance(c.get("data"), dict) else c
    trends = d.get("trends") or []
    b = trends[0].get("buckets", []) if trends else []
    df = pd.DataFrame([{"date": x.get("key_as_string") or x.get("key"), "count": x.get("count", 0)} for x in b])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"].astype(str).str[:19], errors="coerce")
    return df


def stock_prices(entities, date_from, date_to, columns=None):
    """종목 목록의 일별 시세. entities 예: ["KRX:005930", "KRX:000660"] 또는 "KRX:005930".
    가격은 수정주가, 거래량·거래대금·시가총액은 당일 실제 값. change_rate 는 비율(0.01 = 1%)."""
    ents = [entities] if isinstance(entities, str) else list(entities)
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    q = (f'GetStockPrices([{", ".join(ents)}], columns={json.dumps(cols)}, '
         f'date_from={date_from}, date_to={date_to})')
    return ds_df(q)


_UNIVERSE = None


def universe():
    """현재 상장 종목 기본정보(GetStockSymbols 전 종목): symbol, name, market, industry_name 등.
    company_type_l1: 1 일반법인, 2 공기업(한국전력·기업은행 등 8종목), 8 ETF·ETN 등."""
    global _UNIVERSE
    if _UNIVERSE is None:
        u = ds_df("GetStockSymbols()")
        _UNIVERSE = u.rename(columns={"entity_name": "name"})[
            ["symbol", "name", "market", "company_type_l1", "industry_id", "industry_name", "date_listed",
             "fiscal_year_end", "conglomerate_id"]]
    return _UNIVERSE.copy()


def all_stocks(day, common_only=True, columns=None):
    """day(또는 그 이전 마지막 거래일)의 전 종목 하루치 시세 + 종목 정보. 한 번 호출로 4,000여 종목.
    common_only=True 이면 일반법인·공기업 보통주만(ETF·ETN·우선주 제외)."""
    day = pd.Timestamp(day).date()
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    snap = ds_df(f'GetStockPrices(columns={json.dumps(cols)}, '
                 f'date_from={day - timedelta(days=7)}, date_to={day})')
    df = snap.merge(universe().drop(columns=["date"], errors="ignore"), on="symbol", how="left")
    if common_only:
        df = df[df["company_type_l1"].isin(["1", "2"]) & df["symbol"].str.endswith("0")]
    df = df[df["market"].isin(["KOSPI", "KOSDAQ", "KONEX"])]
    if "change_rate" in df:
        df["change_pct"] = df["change_rate"] * 100
    return df.reset_index(drop=True)


def index_prices(date_from, date_to):
    """지수 11종(코스피·코스닥 계열) 일별. change_rate 는 퍼센트(1.94 = 1.94%).
    지수의 value(거래대금)는 백만원, volume(거래량)은 천주 단위로 보입니다(코스닥 2026-11-23 value 4,673,819 ≈ 4.7조)."""
    # 지수를 지정하지 않으면 최근 약 1,300행(≈118거래일)만 오고 앞 기간이 조용히 잘린다 → 140일씩 나눠 받는다
    start, end = pd.Timestamp(date_from), pd.Timestamp(date_to)
    parts = []
    while start <= end:
        stop = min(start + pd.Timedelta(days=139), end)
        parts.append(ds_df(f"GetMarketIndexes(date_from={start.date()}, date_to={stop.date()})"))
        start = stop + pd.Timedelta(days=1)
    df = pd.concat([x for x in parts if not x.empty], ignore_index=True) if parts else pd.DataFrame()
    return df.drop_duplicates(["date", "symbol"]).sort_values(["date", "symbol"]).reset_index(drop=True) if not df.empty else df


def fred(series, date_from, date_to):
    """FRED 시리즈 → 날짜 × 시리즈 넓은 표. 예: fred(["DGS10", "DEXKOUS"], "2026-10-01", "2026-11-24")"""
    ids = ", ".join(json.dumps(s) for s in series)
    df = ds_df(f"QueryFredSeriesData([{ids}], date_from={date_from}, date_to={date_to})")
    if df.empty:
        return df
    val = [c for c in df.columns if c not in ("entity_id", "date")][0]
    df["series"] = df["entity_id"].str.replace("FRED:", "", regex=False)
    wide = df.pivot_table(index="date", columns="series", values=val).sort_index()
    return wide[[s for s in series if s in wide.columns]]  # 요청한 순서대로


def krw(v, digits=1):
    """원 → '138.0조' / '5,027억'."""
    if v is None or pd.isna(v):
        return "-"
    a = abs(v)
    if a >= 1e12:
        return f"{v / 1e12:.{digits}f}조"
    if a >= 1e8:
        return f"{v / 1e8:,.0f}억"
    if a >= 1e4:
        return f"{v / 1e4:,.0f}만"
    return f"{v:,.0f}"


PRICE_NOTICE = ("※ 주가는 수정주가입니다(분할·무상증자·유상증자 소급 반영, 배당 미반영). "
                "거래량·거래대금·시가총액은 당일 실제 값이며 KRX 기준입니다(NXT 미포함).")

# 한글 차트 폰트
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # noqa: F401  (Colab: 위 셀에서 설치)
except ImportError:
    if sys.platform.startswith("win"):
        plt.rcParams["font.family"] = "Malgun Gothic"
    elif sys.platform == "darwin":
        plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

print("DeepSearch 준비 완료", "(Colab)" if IN_COLAB else "(로컬)")

DeepSearch 준비 완료 (로컬)


In [3]:
ASOF = "2026-11-24"
STOCK = "005930"          # 종목별 공시
KEYWORD = "유상증자"       # 키워드별 공시
EXCHANGE = ["시장감시위원회", "유가증권시장본부", "코스닥시장본부", "코넥스시장본부"]

def link_type(url):
    u = str(url or "")
    return "DART" if "dart.fss.or.kr" in u else ("KIND" if "kind.krx.co.kr" in u else "기타")

def doc_pages(category, section, query, date_from, date_to, pages=1, count=100):
    """doc_search 와 같은 표를 주되, 페이지가 count보다 적게 와도 전체 건수에 닿을 때까지 다음 페이지를 받습니다.
    (서버가 중복을 빼고 주는지 한 페이지가 80~90건으로 오는 일이 흔합니다.)"""
    raw, total = [], 0
    for p in range(1, pages + 1):
        q = (f'DocumentSearch(["{category}"],{json.dumps(section, ensure_ascii=False)},{json.dumps(query, ensure_ascii=False)},'
             f'count={count},page={p},date_from={date_from},date_to={date_to})')
        c = _pod(ds(q)).get("content") or {}
        d = c.get("data") if isinstance(c.get("data"), dict) else c
        total = d.get("total_matches") or 0
        docs = d.get("docs") or []
        raw += docs
        if not docs or p * count >= total:
            break
    rows = [{"created_at": pd.to_datetime(str(x.get("created_at"))[:19], errors="coerce"),
             "publisher": x.get("publisher"), "title": " ".join(str(x.get("title") or "").split()),
             "stocks": ", ".join(s.get("name", "") for s in x.get("securities") or []),
             "symbols": ", ".join(s.get("symbol", "") for s in x.get("securities") or []),
             "polarity": _polarity(x), "url": x.get("content_url")} for x in raw]
    cols = ["created_at", "publisher", "title", "stocks", "symbols", "polarity", "url"]
    return total, pd.DataFrame(rows, columns=cols).drop_duplicates("url")

def disclosures(query, date_from, date_to, pages=1):
    total, df = doc_pages("company", ["disclosure"], query, date_from, date_to, pages=pages)
    df["원문"] = df.url.map(link_type)
    print(f"'{query[:80]}' {date_from}~{date_to}: 전체 {total:,}건 중 {len(df)}건")
    return df

## 1. 종목별 — 최근 3개월
종목 지정은 `securities.symbol:6자리`입니다(`KRX:` 접두어를 붙이면 키워드 검색으로 바뀝니다).

In [4]:
d3m = (pd.Timestamp(ASOF) - pd.DateOffset(months=3)).date()
one = disclosures(f"securities.symbol:{STOCK}", d3m, ASOF, pages=2)
print(one.원문.value_counts().to_dict())
one[["created_at", "publisher", "title", "원문", "url"]].head(10)

'securities.symbol:005930' 2026-08-24~2026-11-24: 전체 73건 중 73건
{'DART': 72, 'KIND': 1}


,created_at,publisher,title,원문,url
0,2026-11-18 14:46:34,삼성전자,임원ㆍ주요주주특정증권등소유상황보고서,DART,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20261118000151
1,2026-11-16 16:46:38,삼성전자,분기보고서 (2026.09),DART,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20261116001482
2,2026-11-13 17:16:37,삼성전자,최대주주등소유주식변동신고서,DART,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20261113801935
3,2026-11-13 17:11:37,삼성전자,임원ㆍ주요주주특정증권등소유상황보고서,DART,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20261113001527
4,2026-11-13 16:46:33,삼성전자,동일인등출자계열회사와의상품ㆍ용역거래변경,DART,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20261113001338
5,2026-11-13 16:43:16,삼성전자,동일인등출자계열회사와의상품ㆍ용역거래변경,DART,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20261113001302
6,2026-11-13 16:43:16,삼성전자,동일인등출자계열회사와의상품ㆍ용역거래변경,DART,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20261113001281
7,2026-11-13 15:39:22,삼성전자,기업설명회(IR)개최(안내공시),DART,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20261113801115
8,2026-11-12 09:05:17,삼성전자,임원ㆍ주요주주특정증권등소유상황보고서,DART,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20261112000017
9,2026-11-11 11:12:30,삼성전자,임원ㆍ주요주주특정증권등소유상황보고서,DART,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20261111000113


## 2. 키워드별 — 최근 2주 '유상증자'가 제목에 든 공시
`title:키워드`로 제목만 봅니다. 결정 공시뿐 아니라 정정·발행가 확정 같은 후속 공시도 함께 걸립니다.

In [5]:
d2w = (pd.Timestamp(ASOF) - pd.Timedelta(days=13)).date()
kw = disclosures(f"title:{KEYWORD}", d2w, ASOF, pages=3)
u = universe()
u["code"] = u.symbol.str.replace("KRX:", "", regex=False)
kw["code"] = kw.symbols.str.split(", ").str[0].str.replace("KRX:", "", regex=False)
kw = kw.merge(u[["code", "market"]], on="code", how="left")
kw["유형"] = kw.title.str.replace(r"\s+", "", regex=True).str.extract(r"^(\[[^\]]+\])?(.{0,14})")[1]
print("시장별:", kw.market.fillna("비상장·기타").value_counts().to_dict())
kw.유형.value_counts().head(8)

'title:유상증자' 2026-11-11~2026-11-24: 전체 144건 중 144건


시장별: {'KOSDAQ': 82, 'KOSPI': 28, '비상장·기타': 27, 'KONEX': 7}


유형
주요사항보고서(유상증자결정    69
유상증자결정(종속회사의주요    19
추가상장(유상증자(제3자배    13
특수관계인의유상증자참여      13
유상증자또는주식관련사채등의    12
증권발행결과(자율공시)(제     4
유상증자결정             3
추가상장(유상증자(일반공모     2
Name: count, dtype: int64

In [6]:
kw[["created_at", "stocks", "market", "title", "원문"]].head(10)

,created_at,stocks,market,title,원문
0,2026-11-24 17:13:05,에코프로,KOSDAQ,유상증자결정(종속회사의주요경영사항),DART
1,2026-11-24 17:08:04,에코프로비엠,KOSDAQ,유상증자결정(종속회사의주요경영사항),DART
2,2026-11-24 17:08:04,에코프로비엠,KOSDAQ,[기재정정]유상증자결정(종속회사의주요경영사항),DART
3,2026-11-24 17:08:04,에코프로,KOSDAQ,[기재정정]유상증자결정(종속회사의주요경영사항),DART
4,2026-11-24 17:03:04,국도화학,KOSPI,[기재정정]유상증자결정(종속회사의주요경영사항),DART
5,2026-11-24 17:01:22,한울앤제주,KOSDAQ,추가상장(유상증자(일반공모-소액공모)),KIND
6,2026-11-24 16:58:04,한세엠케이,KOSPI,유상증자또는주식관련사채등의발행결과(자율공시),DART
7,2026-11-24 16:58:04,썬테크,KONEX,주요사항보고서(유상증자결정),DART
8,2026-11-24 16:48:03,서남,KOSDAQ,유상증자또는주식관련사채등의청약결과(자율공시),DART
9,2026-11-24 16:46:46,에코글로우,KOSDAQ,추가상장(유상증자(일반공모-소액공모)),KIND


## 3. 거래소 시장 공시 — 시장감시위원회·시장본부
투자주의·경고·위험 지정, 공매도 과열종목 지정, 단기과열, 매매거래정지, 관리종목 등은 거래소가 냅니다.
`publisher.raw`로 발행기관을 지정합니다. (이 공시들은 2026-08부터 촘촘하게 들어 있습니다.)

In [7]:
pq = "publisher.raw:(" + " or ".join(f"'{p}'" for p in EXCHANGE) + ")"
ex = disclosures(pq, d2w, ASOF, pages=5)
ex["조치"] = (ex.title.str.replace(r"\s+", " ", regex=True)
              .str.replace(r"\(.*?\)|\[.*?\]", "", regex=True).str.strip().str[:22])
print("발행기관:", ex.publisher.value_counts().to_dict())
ex.조치.value_counts().head(12)

'publisher.raw:('시장감시위원회' or '유가증권시장본부' or '코스닥시장본부' or '코넥스시장본부')' 2026-11-11~2026-11-24: 전체 1,011건 중 500건
발행기관: {'코스닥시장본부': 215, '시장감시위원회': 180, '유가증권시장본부': 105}


조치
소수계좌 거래집중 종목      129
공매도 과열종목 지정        51
기타시장안내             30
단기과열종목 지정 연장       22
추가상장               19
특정계좌 매매관여 과다종목     15
공매도 과열종목 연장        13
대량매매내역             12
변경상장               12
추가상장)              11
관리종목지정             10
관리종목지정사유추가          8
Name: count, dtype: int64

In [8]:
ex[ex.title.str.contains("투자경고|공매도 과열", regex=True)][["created_at", "publisher", "stocks", "title"]].head(10)

,created_at,publisher,stocks,title
0,2026-11-24 20:16:07,코스닥시장본부,휴림로봇,공매도 과열종목 지정(공매도 거래 금지 적용)
1,2026-11-24 20:16:04,코스닥시장본부,CJ ENM,공매도 과열종목 지정(공매도 거래 금지 적용)
2,2026-11-24 20:15:37,유가증권시장본부,한솔홈데코,공매도 과열종목 지정(공매도 거래 금지 적용)
3,2026-11-24 20:15:36,유가증권시장본부,,공매도 과열종목 지정(공매도 거래 금지 적용)
4,2026-11-24 20:15:36,코스닥시장본부,큐리언트,공매도 과열종목 지정(공매도 거래 금지 적용)
5,2026-11-24 20:15:36,코스닥시장본부,스튜디오드래곤,공매도 과열종목 지정(공매도 거래 금지 적용)
25,2026-11-24 20:06:05,시장감시위원회,페니트리움바이오,[투자주의]투자경고종목 지정예고
26,2026-11-24 20:06:04,시장감시위원회,썸에이지,투자경고종목지정
42,2026-11-24 20:04:45,시장감시위원회,휴림로봇,[투자주의]투자경고종목 지정예고
48,2026-11-24 20:04:34,시장감시위원회,대한광통신,투자경고종목지정


## 4. 기간 중 공시가 많은 발행기관
문서를 받지 않고 건수만 볼 때는 `DocumentAggregation`을 씁니다(`publisher`로 묶음).

In [9]:
c = ds(f'DocumentAggregation(["company"],["disclosure"],"*","publisher:20",date_from={d2w},date_to={ASOF})')
agg = pd.DataFrame(c["data"]["pods"][1]["content"]["data"])[["key", "count"]]
agg.columns = ["발행기관", "공시 수(2주)"]
agg.head(12)

,발행기관,공시 수(2주)
0,코스닥시장본부,414
1,시장감시위원회,337
2,유가증권시장본부,260
3,미래에셋자산운용,111
4,삼성자산운용,94
5,NH투자증권,80
6,한국투자신탁운용,64
7,교보증권,63
8,하나증권,60
9,한화투자증권,57


## 결과 읽는 법과 한계
- **원문 링크:** DART(`dart.fss.or.kr`) 또는 KIND(`kind.krx.co.kr`)입니다. 본문은 DeepSearch에 없고 링크로 읽어야 합니다(2-05).
- **회사 이름:** 공시의 회사 이름은 DART 등록명(예: '현대자동차')이라 `securities.name:현대차`로는 안 걸립니다. 종목코드로 지정하세요.
- **유형 분류:** 제목 앞부분으로 거칠게 나눈 것입니다. 정확한 공시 유형 코드는 DeepSearch에 없습니다.